# 02 — Model training (Random Forest)

Trains the same **RandomForestClassifier** pipeline as `ml/train_model.py` on `data/processed/final_dataset.csv`, evaluates with a stratified hold-out split, and saves **`ml/model.pkl`** and **`ml/feature_columns.pkl`**.

Run `ml/preprocess.py` first if `final_dataset.csv` is missing.

In [1]:
import os
from pathlib import Path

ROOT = Path.cwd().resolve()
for candidate in [ROOT, ROOT.parent, *ROOT.parents]:
    if (candidate / "data" / "processed" / "final_dataset.csv").is_file():
        os.chdir(candidate)
        break
else:
    raise FileNotFoundError("Run ml/preprocess.py first to create data/processed/final_dataset.csv")
print("CWD:", os.getcwd())

CWD: B:\PBCS\VI Sem\Machine Learning\MiniProject\Final\smart-energy-optimizer


In [2]:
import joblib
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.metrics import classification_report, confusion_matrix

df = pd.read_csv("data/processed/final_dataset.csv")
TARGET = "wastage_label"
FEATURES = [c for c in df.columns if c != TARGET]
X, y = df[FEATURES], df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    class_weight="balanced",
    random_state=42,
)
model.fit(X_train, y_train)

cv_scores = cross_val_score(model, X, y, cv=5, scoring="f1")
print(f"5-fold CV F1: {cv_scores.mean():.3f} ± {cv_scores.std():.3f}")
y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred, target_names=["Normal", "Wastage"]))
print("Confusion matrix:\n", confusion_matrix(y_test, y_pred))

5-fold CV F1: 1.000 ± 0.000
              precision    recall  f1-score   support

      Normal       1.00      1.00      1.00       243
     Wastage       1.00      1.00      1.00       104

    accuracy                           1.00       347
   macro avg       1.00      1.00      1.00       347
weighted avg       1.00      1.00      1.00       347

Confusion matrix:
 [[243   0]
 [  0 104]]


In [3]:
import os
os.makedirs("ml", exist_ok=True)
joblib.dump(model, "ml/model.pkl")
joblib.dump(FEATURES, "ml/feature_columns.pkl")
print("Saved ml/model.pkl and ml/feature_columns.pkl")

Saved ml/model.pkl and ml/feature_columns.pkl
